# ハミルトニアンシミュレーションの回路資源比較

Trotter–Suzuki、QSVT、multiproduct formula (MPF) を、同じ Pauli 和と誤差予算の下で比較します。小規模系では実際の statevector、大規模系では密行列を作らないゲート分解モデルを使います。

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from hamiltonian_resources import (
    BenchmarkConfig, HamiltonianSpec, MultiproductMethod, QSVTMethod,
    TimeScaling, TrotterMethod, build_hamiltonian_qsvt_circuit,
    build_multiproduct_circuit, build_trotter_circuit, compare_with_exact,
    plot_benchmark, run_benchmark, save_benchmark, transverse_field_ising,
)
pd.set_option("display.max_columns", None)

## ベンチマーク設定（ここを編集）

system-size sweepでは相互作用が鎖全体へ広がる時間を比較するため、既定で $t(n)=n$ とします。`TIME_PER_QUBIT`を変更すると $t(n)=\tau n$、`TimeScaling("fixed", t)`に変えると固定時刻になります。

In [ ]:
SYSTEM_SIZES = np.arange(2, 14, 2)
TARGET_ERRORS = np.logspace(-1, -3, 5)
TIME_PER_QUBIT = 1.0

benchmark_config = BenchmarkConfig(
    hamiltonian=HamiltonianSpec(
        model="transverse_field_ising",
        parameters={"coupling": 1.0, "field": 3.0, "periodic": False},
    ),
    system_sizes=SYSTEM_SIZES,
    target_errors=TARGET_ERRORS,
    time=TimeScaling("proportional", TIME_PER_QUBIT),
    fixed_system_size=8,
    fixed_target_error=1e-3,
    methods=[
        *(TrotterMethod(order) for order in (1, 2, 4, 6)),
        *(MultiproductMethod(terms) for terms in (3, 5, 7)),
        QSVTMethod(),
    ],
)

## 1. ハミルトニアン入力

例として $H=-J\sum_i Z_iZ_{i+1}-h\sum_iX_i$ を使います。Qiskit の Pauli label は右端が qubit 0 です。任意の Pauli 和は `PauliHamiltonian.from_terms` で渡せます。

In [ ]:
H = transverse_field_ising(3, coupling=1.0, field=0.7)
print(H.name, H.terms)
print(f"terms={H.term_count}, alpha=sum|h_j|={H.alpha:.3f}")

## 2. 実回路を構成する

下の関数はいずれも Hamiltonian の密行列を指数化しません。QSVTは時刻と許容誤差からcosine/sine位相を合成し、MPFは既定の`new` scheduleでsegmentごとのrobust OAAを構成します。

In [ ]:
time = 0.2
trotter_circuit = build_trotter_circuit(H, time, reps=2, order=2)
mpf_circuit = build_multiproduct_circuit(H, time, m=2, segments=1)
qsvt_circuit = build_hamiltonian_qsvt_circuit(H, time, epsilon=1e-2)

display(trotter_circuit.draw(output="mpl", fold=30))
print("MPF metadata:", mpf_circuit.metadata)
print("QSVT metadata:", qsvt_circuit.metadata)

## 3. 小規模系で厳密解と比較

MPF は branch=0 への postselection 後の fidelity と成功確率を別々に表示します。

In [ ]:
validation = []
for reps in (1, 2, 4, 8):
    row = compare_with_exact(H, time, method="trotter", reps=reps, trotter_order=2)
    row["segments"] = reps
    validation.append(row)
for reps in (1, 2, 4):
    row = compare_with_exact(H, time, method="multiproduct", reps=reps, mpf_m=2)
    row["segments"] = reps
    validation.append(row)
pd.DataFrame(validation)

## 4. notebook内でbenchmarkを実行して可視化

上の設定から両方のsweepをメモリ上で計算します。ファイルは自動生成されません。system-sizeのx軸はlog2、resource軸はlog10です。

data schema、target-error sweep、best-of-family summary、解析上の仮定は`docs/resource_scaling_benchmarks.md`を参照してください。

In [ ]:
benchmark_data = run_benchmark(benchmark_config)
benchmark_data[[
    "sweep", "system_qubits", "evolution_time", "target_error",
    "method_label", "t_count", "cnot_count", "status",
]].head(16)

In [ ]:
figures = (
    plot_benchmark(benchmark_data, sweep="system-size", metric="t_count"),
    plot_benchmark(benchmark_data, sweep="system-size", metric="cnot_count"),
    plot_benchmark(benchmark_data, sweep="target-error", metric="t_count"),
    plot_benchmark(benchmark_data, sweep="target-error", metric="cnot_count"),
)
for figure in figures:
    display(figure)

## 5. 必要な場合だけ保存

Python APIは既定では何も保存しません。結果を残す場合だけ次のセルのコメントを外してください。実行ごとに新しいrun directoryが作られ、以前の結果は上書きされません。

In [ ]:
# run_directory, csv_path, metadata_path = save_benchmark(
#     benchmark_data, benchmark_config, output_root="../benchmark_outputs"
# )
# run_directory

### 解釈上の注意

- QSVT の PREPARE/SELECT コストは Hamiltonian の入力モデルに強く依存します。ここでは一般の Pauli-LCU を公平に数えています。
- 8構成とも決定的動作あたりの比較です。MPF と QSVT の解析値は segment / 回路ごとの 3-step robust OAA を含み、`nominal_success_probability` は 1 です。
- Trotter次数1, 2はChilds交換子上界、次数4, 6はgroup数がwork cap内ならSchubert--Mendl交換子上界を使います。fallbackはCSVの`bound_method`と`bound_rigorous`で明示されます。QSVTの次数は厳密なJacobi–Anger打ち切りです。MPFのsegment数だけは`alpha_eff = min(alpha, W2^(1/3))`による交換子校正proxyであり、小規模では`compare_with_exact`で校正してください。
- 解析モデルの controlled QSVT は、`V`/`V^dagger` が block-encoding query を共有し projector 位相のみ選択する効率的コンパイルを仮定します。
- $t(n)=n$ は伝播速度を厳密に推定したものではなく、局所相互作用が系全体へ広がる時間をサイズに比例させる比較規約です。
- family summaryは各x値で評価済みrowをpost-processし、`selected_method_id`に選択された次数・項数を残します。